# Gold Fact Trade History

- **Purpose**: Transforms `silver.trade_history` into the final `gold.fact_trade_history` table to track trade status lifecycle events (PNDG → SBMT → CMPT). 
- **Business Context**: PWG Pipeline - Trade Domain. This table uses a refined star schema design (direct casting of `SK_TradeID` and preserving the raw `StatusDatetime`) to avoid pipeline-time dimension lookups, pushing dimension resolution to the query layer.
- **Execution Frequency**: Per Batch (Full Rebuild)
- **Inputs**: `silver.trade_history`
- **Outputs**: `gold.fact_trade_history` (CREATE OR REPLACE pattern)
- **Dependencies**: Independent. Can run in parallel with `dim_trade` since it does not require dimension lookups.
- **Expected Row Count**: 3,267,433 (Batch 1)

> Imported our operaitons notebook which include all the functions and all

In [0]:
%run ../../02_common_utils/operations

In [0]:
# configuring widgets for ease of use and reusability
dbutils.widgets.text("env_catalog", "charles_schwab_retailbrokerage_dev_team_lemma")
dbutils.widgets.text("batch_id","1")

batch_id = dbutils.widgets.get("batch_id")
catalog = dbutils.widgets.get("env_catalog")


silver_tbl = f"{catalog}.silver.trade_history"
gold_tbl = f"{catalog}.gold.fact_trade_history"

In [0]:
# importing functions and libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Reading from source table which is silver_trade_history
g_df = spark.read.table(silver_tbl)
g_df.cache()
# storing the count of records in the source table
source_count = g_df.count()

In [0]:
# Logging functions for initial load
l_df = g_df.orderBy(col('_load_ts').desc()).limit(1)
carried_run_id = str(l_df.select("_run_id").first()[0])

# Actual functions to log came from operations framework
log_pipeline_message(spark, carried_run_id, 'INFO', 'gold_fact_trade_history', 'Starting processing for standalone gold fact_trade_history')
start_pipeline_run(spark, carried_run_id, batch_id)
log_domain_run_status(spark, carried_run_id, batch_id, 'TRADE', 'RUNNING')

In [0]:
try:
    # Transforming source columns and adding gold schema fields for fact_trade_history
    g_df1 = g_df.withColumns({
        "SK_TradeID" : col("TH_T_ID").cast("bigint"),
        "StatusDatetime" : col("TH_DTS"),
        "StatusCode" : col("TH_ST_ID"),
        "_load_ts" : current_timestamp()
    })

    g_df2 = g_df1.select("SK_TradeID","StatusDatetime","StatusCode","_batch","_load_ts")
except Exception as e:
    print(f"Error during transformation: {e}")
    raise

In [0]:
# Writing to gold.fact_trade_history table
g_df2.write.format("delta").mode("overwrite").saveAsTable(gold_tbl)

In [0]:
# Using history to get final row count as this is less expensive then count(), coz it doesnt need to rescan the table.
target_count = spark.sql(f"describe history {gold_tbl}").select("operationMetrics.numOutputRows").take(1)[0][0]

null_count = g_df2.filter(col("SK_TradeID").isNull()).count()

log_dq_result(spark, carried_run_id, gold_tbl, "Null SK_TradeID Check", null_count, source_count)
log_domain_run_status(spark, carried_run_id, batch_id, 'TRADE', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'gold_fact_trade_history', 'Successfully completed standalone gold fact_trade_history')
log_gold_recon(spark, carried_run_id, gold_tbl, expected_count=3267433, actual_count=int(target_count))

In [0]:
try:
    # Operations Logging
    # Extract the carry-forwarded _run_id from dataframe
    carried_run_id = str(g_df.select("_run_id").first()[0])

    log_pipeline_recon(
        spark=spark,
        run_id=carried_run_id,
        batch_id=batch_id,
        domain="TRADE",
        table_name="fact_trade_history",
        source_layer="silver",
        target_layer="gold",
        source_count=int(source_count),
        target_count=int(target_count)
    )

    log_audit_event(
        spark=spark,
        run_id=carried_run_id,
        batch=batch_id,
        layer="gold",
        table_name="fact_trade_history",
        operation="OVERWRITE",
        rows_affected=int(target_count)
    )
    print("done")
    
except Exception as e:
    print(f"Error during operations logging: {e}")
    raise